In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy

TIME_OFFSET = 10800 #ADT to UTC is 3hrs

In [2]:
def entropy_feature(x):
    counts = x.value_counts()
    if len(counts) <= 1:
        return 0.0
    return entropy(counts)

def coeff_var(x):
    mean = x.mean()
    if mean == 0:
        return 0.0
    return x.std() / mean

In [3]:
import pandas as pd
import numpy as np

def process_file(k, input_csv, output_csv):
    print(f"Processing {input_csv}...")

    df = pd.read_csv(input_csv)
    df["timestamp"] = df["frame.time_epoch"] + TIME_OFFSET
    df = df.sort_values("timestamp")
    df["time_window"] = (df["timestamp"] * k).astype(int)

    has_transaction_id = "modbus.transaction_id" in df.columns

    temp_src = df["ip.src"].dropna().astype(str)
    temp_dst = df["ip.dst"].dropna().astype(str)

    # ── Discover all IPs in this file ──────────────────────────────────────────
    known_ips = sorted(set(temp_src.unique()) | set(temp_dst.unique()))
    print(f"IPs found: {known_ips}")

    # Fix IAT: compute within (time_window, src, dst) to avoid cross-window bleed
    df["iat"] = (
        df.groupby(["time_window", "ip.src", "ip.dst"])["timestamp"]
        .diff()
        .fillna(0)
    )


    # ── Full contiguous time window index ─────────────────────────────────────
    tw_min, tw_max = df["time_window"].min(), df["time_window"].max()
    full_index = pd.RangeIndex(tw_min, tw_max + 1, name="time_window")

    # ── Per-IP feature extraction ─────────────────────────────────────────────
    ip_frames = []

    for ip in known_ips:
        tx = df[df["ip.src"] == ip]
        rx = df[df["ip.dst"] == ip]

        tx_agg = tx.groupby("time_window").agg(
            tx_packet_count   = ("frame.len", "count"),
            tx_total_bytes    = ("frame.len", "sum"),
            tx_mean_pkt_size  = ("frame.len", "mean"),
            tx_std_pkt_size   = ("frame.len", "std"),
            tx_unique_dst     = ("ip.dst",    "nunique"),
            tx_iat_mean       = ("iat",       "mean"),
            tx_iat_std        = ("iat",       "std"),
            tx_iat_cv         = ("iat",       coeff_var),
            tx_burstiness     = ("iat",       lambda x: x.max() - x.min()),
            tx_unique_fc      = ("modbus.func_code", "nunique"),
            tx_func_entropy   = ("modbus.func_code", entropy_feature),
            tx_read_count     = ("modbus.func_code", lambda x: x.isin([3, 4]).sum()),
            tx_write_count    = ("modbus.func_code", lambda x: x.isin([5, 6, 15, 16]).sum()),
            tx_fc3            = ("modbus.func_code", lambda x: (x == 3).sum()),
            tx_fc4            = ("modbus.func_code", lambda x: (x == 4).sum()),
            tx_fc6            = ("modbus.func_code", lambda x: (x == 6).sum()),
            tx_fc16           = ("modbus.func_code", lambda x: (x == 16).sum()),
            tx_exception_count= ("modbus.exception_code", lambda x: x.notna().sum()),
            tx_unique_regs    = ("modbus.reference_num", "nunique"),
            tx_register_mean  = ("modbus.reference_num", "mean"),
            tx_register_std   = ("modbus.reference_num", "std"),
            tx_register_entropy=("modbus.reference_num", entropy_feature),
            tx_regval_mean    = ("modbus.regval_uint16", "mean"),
            tx_regval_std     = ("modbus.regval_uint16", "std"),
            tx_word_cnt_mean  = ("modbus.word_cnt", "mean"),
            tx_byte_cnt_mean  = ("modbus.byte_cnt", "mean"),
            tx_resp_time_mean = ("modbus.response_time", "mean"),
            tx_resp_time_std  = ("modbus.response_time", "std"),
        )

        rx_agg = rx.groupby("time_window").agg(
            rx_packet_count   = ("frame.len", "count"),
            rx_total_bytes    = ("frame.len", "sum"),
            rx_unique_src     = ("ip.src",    "nunique"),
        )

        if has_transaction_id:
            tx_tid = tx.groupby("time_window").agg(
                tx_unique_tids = ("modbus.transaction_id", "nunique")
            )
            tx_agg = tx_agg.join(tx_tid, how="left")

        # Reindex to full window range, fill silence with 0
        tx_agg = tx_agg.reindex(full_index, fill_value=0)
        rx_agg = rx_agg.reindex(full_index, fill_value=0)

        ip_df = tx_agg.join(rx_agg, how="left").fillna(0)

        # Derived ratios (safe division already guaranteed since fillna(0))
        ip_df["tx_write_ratio"]     = np.where(ip_df["tx_packet_count"] > 0, ip_df["tx_write_count"] / ip_df["tx_packet_count"], 0.0)
        ip_df["tx_read_ratio"]      = np.where(ip_df["tx_packet_count"] > 0, ip_df["tx_read_count"]  / ip_df["tx_packet_count"], 0.0)
        ip_df["tx_exception_ratio"] = np.where(ip_df["tx_packet_count"] > 0, ip_df["tx_exception_count"] / ip_df["tx_packet_count"], 0.0)

        if has_transaction_id:
            ip_df["tx_dup_tids"]    = ip_df["tx_packet_count"] - ip_df["tx_unique_tids"]
            ip_df["tx_tid_ratio"]   = np.where(ip_df["tx_packet_count"] > 0, ip_df["tx_unique_tids"] / ip_df["tx_packet_count"], 0.0)
        else:
            ip_df["tx_unique_tids"] = 0
            ip_df["tx_dup_tids"]    = 0
            ip_df["tx_tid_ratio"]   = 0.0

        # Temporal delta on register values (now computed on gap-filled sequence)
        ip_df["tx_regval_delta"] = ip_df["tx_regval_mean"].diff().fillna(0)

        # Prefix all columns with the IP
        safe_ip = ip.replace(".", "_")
        ip_df = ip_df.add_prefix(f"{safe_ip}__")

        ip_frames.append(ip_df)

    # ── Join all IPs into one row per time_window ─────────────────────────────
    result = pd.DataFrame(index=full_index)
    for ip_df in ip_frames:
        result = result.join(ip_df, how="left")

    result = result.fillna(0).reset_index()

    packet_counts = df.groupby("time_window").size()
    print("Window packet count description:")
    print(packet_counts.describe())

    result.to_csv(output_csv, index=False)
    print(f"Saved → {output_csv}\n")

In [4]:
process_file(1, "../train/benign_nw.csv", "../train/1s_benign_flows.csv")

Processing ../train/benign_nw.csv...
IPs found: ['185.175.0.1', '185.175.0.3', '185.175.0.4', '185.175.0.5', '185.175.0.6', '185.175.0.8', '224.0.0.22', '224.0.0.251']


/tmp/ipykernel_321867/1589412899.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result = result.fillna(0).reset_index()


Window packet count description:
count    7085.000000
mean       75.892025
std        90.031639
min         1.000000
25%        50.000000
50%        50.000000
75%        90.000000
max      1292.000000
dtype: float64
Saved → ../train/1s_benign_flows.csv



In [5]:
process_file(1, "../train/cscada_attack_ssw.csv", "../train/1s_cscada_flows.csv")

Processing ../train/cscada_attack_ssw.csv...
IPs found: ['185.175.0.1', '185.175.0.3', '185.175.0.4', '185.175.0.5', '185.175.0.6', '185.175.0.8', '224.0.0.251']


/tmp/ipykernel_321867/1589412899.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result = result.fillna(0).reset_index()


Window packet count description:
count    14981.000000
mean        77.228289
std        144.291295
min          1.000000
25%         45.000000
50%         50.000000
75%         90.000000
max       8241.000000
dtype: float64
Saved → ../train/1s_cscada_flows.csv



In [6]:
process_file(1, "../train/ext_attack_nw.csv", "../train/1s_external_flows.csv")

Processing ../train/ext_attack_nw.csv...
IPs found: ['185.175.0.1', '185.175.0.3', '185.175.0.4', '185.175.0.5', '185.175.0.6', '185.175.0.7', '185.175.0.8', '224.0.0.22', '224.0.0.251']


/tmp/ipykernel_321867/1589412899.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result = result.fillna(0).reset_index()


Window packet count description:
count     1881.000000
mean       290.755981
std       1252.626481
min          1.000000
25%         45.000000
50%         45.000000
75%         51.000000
max      20281.000000
dtype: float64
Saved → ../train/1s_external_flows.csv



In [7]:
df = pd.read_csv("../train/1s_cscada_flows.csv")
print(df.columns)

Index(['time_window', '185_175_0_1__tx_packet_count',
       '185_175_0_1__tx_total_bytes', '185_175_0_1__tx_mean_pkt_size',
       '185_175_0_1__tx_std_pkt_size', '185_175_0_1__tx_unique_dst',
       '185_175_0_1__tx_iat_mean', '185_175_0_1__tx_iat_std',
       '185_175_0_1__tx_iat_cv', '185_175_0_1__tx_burstiness',
       ...
       '224_0_0_251__rx_packet_count', '224_0_0_251__rx_total_bytes',
       '224_0_0_251__rx_unique_src', '224_0_0_251__tx_write_ratio',
       '224_0_0_251__tx_read_ratio', '224_0_0_251__tx_exception_ratio',
       '224_0_0_251__tx_unique_tids', '224_0_0_251__tx_dup_tids',
       '224_0_0_251__tx_tid_ratio', '224_0_0_251__tx_regval_delta'],
      dtype='str', length=267)


In [8]:
# process_file(10, "../train/benign_nw.csv", "../train/100ms_benign_flows.csv") #this is for one second windows

In [9]:
# process_file(10, "../train/cscada_attack_ssw.csv", "../train/100ms_cscada_flows.csv")

In [10]:
# process_file(10, "../train/ext_attack_nw.csv", "../train/100ms_external_flows.csv")

In [11]:
df = pd.read_csv("../train/1s_cscada_flows.csv")
for col in df.columns:
    print("\"" + col + "\",", end=" ")

"time_window", "185_175_0_1__tx_packet_count", "185_175_0_1__tx_total_bytes", "185_175_0_1__tx_mean_pkt_size", "185_175_0_1__tx_std_pkt_size", "185_175_0_1__tx_unique_dst", "185_175_0_1__tx_iat_mean", "185_175_0_1__tx_iat_std", "185_175_0_1__tx_iat_cv", "185_175_0_1__tx_burstiness", "185_175_0_1__tx_unique_fc", "185_175_0_1__tx_func_entropy", "185_175_0_1__tx_read_count", "185_175_0_1__tx_write_count", "185_175_0_1__tx_fc3", "185_175_0_1__tx_fc4", "185_175_0_1__tx_fc6", "185_175_0_1__tx_fc16", "185_175_0_1__tx_exception_count", "185_175_0_1__tx_unique_regs", "185_175_0_1__tx_register_mean", "185_175_0_1__tx_register_std", "185_175_0_1__tx_register_entropy", "185_175_0_1__tx_regval_mean", "185_175_0_1__tx_regval_std", "185_175_0_1__tx_word_cnt_mean", "185_175_0_1__tx_byte_cnt_mean", "185_175_0_1__tx_resp_time_mean", "185_175_0_1__tx_resp_time_std", "185_175_0_1__rx_packet_count", "185_175_0_1__rx_total_bytes", "185_175_0_1__rx_unique_src", "185_175_0_1__tx_write_ratio", "185_175_0_1__tx